In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd '/content/drive/MyDrive/Climate Mitigation Classification'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Climate Mitigation Classification


In [2]:
import numpy as np
import pandas as pd

In [ ]:
model_dir='final_model_silver_pretrained_gold_finetuned'

In [3]:
#ep=pd.read_csv('/content/drive/MyDrive/Syncpol/Data/cleaned_data/')
com=pd.read_feather("/content/drive/MyDrive/Syncpol/Data/cleaned_data_final/Commission_corpus_translated.feather")
#com=pd.read_csv('/content/drive/MyDrive/Climate Mitigation Classification/climate_classification/commission_climate_classification_results.csv')
#council=pd.read_csv('/content/drive/MyDrive/Climate Mitigation Classification/Climate Mitigation Council Sentences/council_climate_classification_results.csv')

In [4]:
com = com.rename(columns={'sentence_text': 'sentence'})


In [7]:
# --------------------------
#  Tokenization
# --------------------------
from transformers import AutoTokenizer # Import AutoTokenizer here
from datasets import Dataset # Import Dataset from the datasets library

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["sentence"], padding="max_length", truncation=True, max_length=256)

'''# Tokenize all datasets
print("Tokenizing datasets...")
# Convert the pandas DataFrame to a Hugging Face Dataset
com_dataset = Dataset.from_pandas(com)
com_dataset = com_dataset.map(tokenize, batched=True)
com_dataset = com_dataset.remove_columns(["sentence"])'''

AttributeError: partially initialized module 'torch' has no attribute '_subclasses' (most likely due to a circular import)

In [ ]:
# @title
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os
from tqdm.auto import tqdm # Import tqdm
import pyarrow.feather as feather # Import feather
from datasets import Dataset # Import Dataset here

model_dir='final_model_silver_pretrained_gold_finetuned'
model_name = "roberta-base" # Use the correct model name

# Load the model
if os.path.isdir(model_dir):
  model = AutoModelForSequenceClassification.from_pretrained(model_dir)

  # Process in chunks to avoid memory issues
  chunk_size = 100000 # Adjust chunk size as needed
  all_predictions = []

  # Get the total number of rows to calculate the number of chunks
  total_rows = len(com)
  num_chunks = (total_rows + chunk_size - 1) // chunk_size

  print(f"Processing data in {num_chunks} chunks...")

  # Iterate through the DataFrame in chunks
  for i in tqdm(range(0, total_rows, chunk_size), total=num_chunks, desc="Processing Chunks"):
      chunk_df = com[i : i + chunk_size]

      # Convert the pandas DataFrame chunk to a Hugging Face Dataset
      chunk_dataset = Dataset.from_pandas(chunk_df)
      chunk_dataset = chunk_dataset.map(tokenize, batched=True)
      chunk_dataset = chunk_dataset.remove_columns(["sentence"]) # Assuming 'sentence' is the text column

      # Process in batches within the chunk
      batch_size = 256 #16 # Adjust batch size as needed
      predictions = []

      # Iterate through the tokenized dataset chunk in batches with a progress bar
      for j in tqdm(range(0, len(chunk_dataset), batch_size), leave=False, desc=f"Processing Chunk {i//chunk_size + 1}/{num_chunks}"): # Wrap with tqdm and update description
          batch = chunk_dataset[j : j + batch_size]
          # Extract input_ids and attention_mask as tensors
          input_ids = torch.tensor(batch['input_ids'])
          attention_mask = torch.tensor(batch['attention_mask'])

          with torch.no_grad():
              outputs = model(input_ids, attention_mask=attention_mask)

          batch_predictions = torch.argmax(outputs.logits, dim=1).numpy()
          predictions.extend(batch_predictions)

      all_predictions.extend(predictions)

  # Add predictions to the DataFrame
  com['predicted_class'] = all_predictions

  display(com.head())

else:
  print(f"Error: Model directory not found at {model_dir}")
  print("Please ensure the 'model_dir' variable points to a valid directory containing the model files.")

Processing data in 18 chunks...


Processing Chunks:   0%|          | 0/18 [00:00<?, ?it/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Processing Chunk 1/18:   0%|          | 0/6250 [00:00<?, ?it/s]

In [ ]:
# @title
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os
from tqdm.auto import tqdm
import pyarrow.feather as feather
from datasets import Dataset
from torch.utils.data import DataLoader
import gc

model_dir = 'final_model_silver_pretrained_gold_finetuned'
model_name = "roberta-base"

# Load the model
if os.path.isdir(model_dir):
    # 1. Use GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()  # Set to evaluation mode

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Define tokenization function
    def tokenize(examples):
        return tokenizer(
            examples["sentence"],
            padding="max_length",
            truncation=True,
            max_length=512
        )

    # 2. Use larger chunks and batch size
    chunk_size = 500000  # Increase chunk size (adjust if OOM)
    batch_size = 256  # Large batch size for GPU (adjust if OOM)

    # Get the total number of rows
    total_rows = len(com)
    num_chunks = (total_rows + chunk_size - 1) // chunk_size

    print(f"Processing {total_rows} rows in {num_chunks} chunks with batch size {batch_size}...")

    all_predictions = []

    # Iterate through the DataFrame in chunks
    for i in tqdm(range(0, total_rows, chunk_size), total=num_chunks, desc="Processing Chunks"):
        chunk_df = com.iloc[i : i + chunk_size]

        # Convert to Dataset and tokenize
        chunk_dataset = Dataset.from_pandas(chunk_df)
        chunk_dataset = chunk_dataset.map(tokenize, batched=True, batch_size=1000)
        chunk_dataset = chunk_dataset.remove_columns(["sentence"])

        # Set format for PyTorch
        chunk_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])

        # Use DataLoader for efficient batching
        dataloader = DataLoader(chunk_dataset, batch_size=batch_size, shuffle=False)

        # Process with GPU
        with torch.no_grad():
            for batch in dataloader:
                # Move batch to device
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)

                # Forward pass
                outputs = model(input_ids, attention_mask=attention_mask)

                # Get predictions and move back to CPU immediately
                predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                all_predictions.extend(predictions)

        # Clear memory after each chunk
        del chunk_df, chunk_dataset, dataloader
        gc.collect()
        torch.cuda.empty_cache()

    # Add predictions to the DataFrame
    com['predicted_class'] = all_predictions

    display(com.head())

else:
    print(f"Error: Model directory not found at {model_dir}")
    print("Please ensure the 'model_dir' variable points to a valid directory containing the model files.")

Using device: cuda
Processing 1766330 rows in 4 chunks with batch size 256...


Processing Chunks:   0%|          | 0/4 [00:00<?, ?it/s]

Map:   0%|          | 0/500000 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [5]:
#*****
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os
from tqdm.auto import tqdm
import pyarrow.feather as feather
from datasets import Dataset # Import Dataset here
from torch.utils.data import DataLoader
import gc

model_dir = 'final_model_silver_pretrained_gold_finetuned'
model_name = "roberta-base"

# Load the model
if os.path.isdir(model_dir):
    # 1. Use GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()  # Set to evaluation mode

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Define tokenization function (added here)
    def tokenize(examples):
        return tokenizer(
            examples["sentence"],
            padding="max_length",
            truncation=True,
            max_length=512
        )

    # 2. Use larger chunks and batch size
    chunk_size = 500000  # Increase chunk size (adjust if OOM)
    batch_size = 256  # Large batch size for GPU (adjust if OOM)

    # Get the total number of rows
    total_rows = len(com)
    num_chunks = (total_rows + chunk_size - 1) // chunk_size

    print(f"Processing {total_rows} rows in {num_chunks} chunks with batch size {batch_size}...")

    all_predictions = []

    # Create checkpoint directory
    checkpoint_dir = 'inference_checkpoints'
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Check if there are existing checkpoints to resume from
    existing_checkpoints = sorted([f for f in os.listdir(checkpoint_dir) if f.startswith('chunk_')])
    start_chunk = 0

    if existing_checkpoints:
        last_checkpoint = existing_checkpoints[-1]
        start_chunk = int(last_checkpoint.split('_')[1].split('.')[0]) + 1
        print(f"Resuming from chunk {start_chunk}...")

        # Load existing predictions
        import pickle
        for checkpoint_file in existing_checkpoints:
            with open(os.path.join(checkpoint_dir, checkpoint_file), 'rb') as f:
                chunk_predictions = pickle.load(f)
                all_predictions.extend(chunk_predictions)

    # Iterate through the DataFrame in chunks
    for chunk_idx, i in enumerate(tqdm(range(start_chunk * chunk_size, total_rows, chunk_size),
                                        initial=start_chunk,
                                        total=num_chunks,
                                        desc="Processing Chunks")):
        # Define chunk_number within the loop
        chunk_number = start_chunk + chunk_idx
        chunk_df = com.iloc[i : i + chunk_size]

        # Convert to Dataset and tokenize
        chunk_dataset = Dataset.from_pandas(chunk_df)
        chunk_dataset = chunk_dataset.map(tokenize, batched=True, batch_size=1000)
        chunk_dataset = chunk_dataset.remove_columns(["sentence"])

        # Set format for PyTorch
        chunk_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])

        # Use DataLoader for efficient batching
        dataloader = DataLoader(chunk_dataset, batch_size=batch_size, shuffle=False)

        # Store chunk predictions separately
        chunk_predictions = []

        # Process with GPU
        with torch.no_grad():
            for batch in tqdm(dataloader,
                            desc=f"Chunk {chunk_number + 1}/{num_chunks}", # Corrected chunk number display
                            leave=False):
                # Move batch to device
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)

                # Forward pass
                outputs = model(input_ids, attention_mask=attention_mask)

                # Get predictions and move back to CPU immediately
                predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                chunk_predictions.extend(predictions)

        # Add chunk predictions to all predictions
        all_predictions.extend(chunk_predictions)

        # Save checkpoint after each chunk
        checkpoint_file = os.path.join(checkpoint_dir, f'chunk_{chunk_number}.pkl')
        import pickle
        with open(checkpoint_file, 'wb') as f:
            pickle.dump(chunk_predictions, f)
        print(f"Saved checkpoint: {checkpoint_file}")

        # Clear memory after each chunk
        del chunk_df, chunk_dataset, dataloader, chunk_predictions
        gc.collect()
        torch.cuda.empty_cache()

    # Add predictions to the DataFrame
    com['predicted_class'] = all_predictions

    # Save final results
    output_file = 'com_with_predictions.feather'
    feather.write_feather(com, output_file)
    print(f"\nFinal results saved to: {output_file}")

    # Optional: Clean up checkpoint files after successful completion
    import shutil
    shutil.rmtree(checkpoint_dir)
    print(f"Checkpoint directory '{checkpoint_dir}' removed after successful completion.")

    display(com.head())

else:
    print(f"Error: Model directory not found at {model_dir}")
    print("Please ensure the 'model_dir' variable points to a valid directory containing the model files.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

Processing Chunks:  75%|#######5  | 3/4 [00:00<?, ?it/s]

Map:   0%|          | 0/266330 [00:00<?, ? examples/s]

Chunk 4/4:   0%|          | 0/1041 [00:00<?, ?it/s]

Saved checkpoint: inference_checkpoints/chunk_3.pkl

Final results saved to: com_with_predictions.feather
Checkpoint directory 'inference_checkpoints' removed after successful completion.


,speech_id,sentence_id,title,date,context,speaker,url,pdfurl,langdetect,sentence,...,sentence_text_en,pdfonly,after.sampling,text.annotated,president,vice_president,commissioner,migration_vp,commission,predicted_class
0,29085.0,1,Opening remarks by Commissioner Hoekstra at th...,2024-11-18,Baku,wopke hoekstra,https://ec.europa.eu/commission/presscorner/de...,https://ec.europa.eu/commission/presscorner/ap...,en,"Thank you , Ladies and gentlemen ,",...,"Thank you , Ladies and gentlemen ,",FALSE,True,None,NO,NO,YES,NO,Von der Leyen,0
1,29085.0,2,Opening remarks by Commissioner Hoekstra at th...,2024-11-18,Baku,wopke hoekstra,https://ec.europa.eu/commission/presscorner/de...,https://ec.europa.eu/commission/presscorner/ap...,en,Truly happy to be here for my second COP as EU...,...,Truly happy to be here for my second COP as EU...,FALSE,True,None,NO,NO,YES,NO,Von der Leyen,0
2,29085.0,3,Opening remarks by Commissioner Hoekstra at th...,2024-11-18,Baku,wopke hoekstra,https://ec.europa.eu/commission/presscorner/de...,https://ec.europa.eu/commission/presscorner/ap...,en,I am intimately familiar with the fact that at...,...,I am intimately familiar with the fact that at...,FALSE,True,None,NO,NO,YES,NO,Von der Leyen,0
3,29085.0,4,Opening remarks by Commissioner Hoekstra at th...,2024-11-18,Baku,wopke hoekstra,https://ec.europa.eu/commission/presscorner/de...,https://ec.europa.eu/commission/presscorner/ap...,en,Let 's address the elephant in the room right ...,...,Let 's address the elephant in the room right ...,FALSE,True,None,NO,NO,YES,NO,Von der Leyen,0
4,29085.0,5,Opening remarks by Commissioner Hoekstra at th...,2024-11-18,Baku,wopke hoekstra,https://ec.europa.eu/commission/presscorner/de...,https://ec.europa.eu/commission/presscorner/ap...,en,But then again when has it ever been easy ?,...,But then again when has it ever been easy ?,FALSE,True,None,NO,NO,YES,NO,Von der Leyen,0


In [ ]:
#*****
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os
from tqdm.auto import tqdm
import pyarrow.feather as feather
from datasets import Dataset # Import Dataset here
from torch.utils.data import DataLoader
import gc

model_dir = 'final_model_silver_pretrained_gold_finetuned'
model_name = "roberta-base"

# Load the model
if os.path.isdir(model_dir):
    # 1. Use GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()  # Set to evaluation mode

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Define tokenization function (added here)
    def tokenize(examples):
        return tokenizer(
            examples["sentence"],
            padding="max_length",
            truncation=True,
            max_length=512
        )

    # 2. Use larger chunks and batch size
    chunk_size = 500000  # Increase chunk size (adjust if OOM)
    batch_size = 256  # Large batch size for GPU (adjust if OOM)

    # Get the total number of rows
    total_rows = len(com)
    num_chunks = (total_rows + chunk_size - 1) // chunk_size

    print(f"Processing {total_rows} rows in {num_chunks} chunks with batch size {batch_size}...")

    all_predictions = []

    # Create checkpoint directory
    checkpoint_dir = 'inference_checkpoints'
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Check if there are existing checkpoints to resume from
    existing_checkpoints = sorted([f for f in os.listdir(checkpoint_dir) if f.startswith('chunk_')])
    start_chunk = 0

    if existing_checkpoints:
        last_checkpoint = existing_checkpoints[-1]
        start_chunk = int(last_checkpoint.split('_')[1].split('.')[0]) + 1
        print(f"Resuming from chunk {start_chunk}...")

        # Load existing predictions
        import pickle
        for checkpoint_file in existing_checkpoints:
            with open(os.path.join(checkpoint_dir, checkpoint_file), 'rb') as f:
                chunk_predictions = pickle.load(f)
                all_predictions.extend(chunk_predictions)

    # Iterate through the DataFrame in chunks
    for chunk_idx, i in enumerate(tqdm(range(start_chunk * chunk_size, total_rows, chunk_size),
                                        initial=start_chunk,
                                        total=num_chunks,
                                        desc="Processing Chunks")):
        # Define chunk_number within the loop
        chunk_number = start_chunk + chunk_idx
        chunk_df = com.iloc[i : i + chunk_size]

        # Convert to Dataset and tokenize
        chunk_dataset = Dataset.from_pandas(chunk_df)
        chunk_dataset = chunk_dataset.map(tokenize, batched=True, batch_size=1000)
        chunk_dataset = chunk_dataset.remove_columns(["sentence"])

        # Set format for PyTorch
        chunk_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])

        # Use DataLoader for efficient batching
        dataloader = DataLoader(chunk_dataset, batch_size=batch_size, shuffle=False)

        # Store chunk predictions separately
        chunk_predictions = []

        # Process with GPU
        with torch.no_grad():
            num_batches = len(dataloader)
            for batch_idx, batch in enumerate(tqdm(dataloader,
                            desc=f"Chunk {chunk_number + 1}/{num_chunks}", # Corrected chunk number display
                            leave=False)):
                # Move batch to device
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)

                # Forward pass
                outputs = model(input_ids, attention_mask=attention_mask)

                # Get predictions and move back to CPU immediately
                predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                chunk_predictions.extend(predictions)

                # Save checkpoint every 10% of batches within the chunk
                if (batch_idx + 1) % (num_batches // 10) == 0 or (batch_idx + 1) == num_batches:
                    checkpoint_file = os.path.join(checkpoint_dir, f'chunk_{chunk_number}_batch_{batch_idx}.pkl')
                    import pickle
                    with open(checkpoint_file, 'wb') as f:
                        pickle.dump(chunk_predictions, f)
                    print(f"Saved checkpoint: {checkpoint_file}")


        # Add chunk predictions to all predictions
        all_predictions.extend(chunk_predictions)


        # Clear memory after each chunk
        del chunk_df, chunk_dataset, dataloader, chunk_predictions
        gc.collect()
        torch.cuda.empty_cache()

    # Add predictions to the DataFrame
    com['predicted_class'] = all_predictions

    # Save final results
    output_file = 'com_with_predictions.feather'
    feather.write_feather(com, output_file)
    print(f"\nFinal results saved to: {output_file}")

    # Optional: Clean up checkpoint files after successful completion
    import shutil
    shutil.rmtree(checkpoint_dir)
    print(f"Checkpoint directory '{checkpoint_dir}' removed after successful completion.")

    display(com.head())

else:
    print(f"Error: Model directory not found at {model_dir}")
    print("Please ensure the 'model_dir' variable points to a valid directory containing the model files.")